In [ ]:
import sys
import os

import h5py

from matplotlib import pyplot as plt

import numpy as np
import pandas as pd
import scipy as sp

from sklearn.decomposition import PCA


sys.path.append(os.path.join(os.getcwd(), '../../'))

# noinspection PyUnresolvedReferences
from utils import read_utils
# noinspection PyUnresolvedReferences
from utils import plot_utils
# noinspection PyUnresolvedReferences
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
read_ids = list(range(0, 12+1))
# read_ids = [0, 2, 3, 8, 9]
# read_ids = [6]
plot_id = 6

dims = 3

# selection_n = ['DD01', 'VB02', 'SMDDR', 'AVAR']
# selection_bhl = ['undefined', 'quiescence', 'ab_bending', 'grooming_b', 'grooming_f', 'running']

In [ ]:
path = os.path.join(read_utils.get_data_path(), 'fly-2023-schaffer/brain')

(file_ids, filenames,
 traces,
 states, markers) = read_utils.get_data(path, read_ids)

# # clear nan values
# for i in range(len(file_ids)):
# 	mask = markers[i].notna().all(axis=1)
#
# 	traces[i] = traces[i][mask]
# 	markers[i] = markers[i][mask]
# 	states[i] = states[i][mask]
#
# 	#filenames[i] = filenames[i][mask]

# # remove unused discrete state labels
# states = [ds[selection_bhl] for ds in states]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep
def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(data=trace.values[1:, :], columns=trace.columns) for trace in traces]
traces = [pd.DataFrame(highpass_butter(trace.values, sr, 1 / 100), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

states = [pd.DataFrame(data=state.values[1:, :], columns=state.columns) for state in states]

markers = [pd.DataFrame(data=marker.values[1:, :] - marker.values[:-1, :], columns=marker.columns) for marker in markers]
markers = [
	pd.DataFrame(
		data=(marker.values - marker.values.mean(axis=0)) / np.where((marker_std := np.std(marker.values, axis=0)) == 0.0, 1.0, marker_std),
		columns=marker.columns)
	for marker in markers
]

In [ ]:
(modes_manifold,
 recos_manifold,
 loads_pca_man,
 vars_expl_mans) = train_utils.get_manifolds_pca(traces,
												 n_pc=dims)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_manifold, states,
						  [vx[:2] for vx in vars_expl_mans], None,
						  title='PCA manifolds', top_space=2.5, save_name=None)

In [ ]:
# plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 						  modes_manifold, states,
# 						  [vx[:3] for vx in vars_expl_mans], None,
# 						  title='PCA manifolds', top_space=3.5)

In [ ]:
traces_manifolds_indices = [np.argsort(xloads_pca_man[0, :]) for xloads_pca_man in loads_pca_man]

traces_sorted = [t.iloc[:, t_man_idx.tolist()] for t, t_man_idx in zip(traces, traces_manifolds_indices)]
traces_manifolds_sorted = [t_man.iloc[:, t_man_idx.tolist()] for t_man, t_man_idx in zip(recos_manifold, traces_manifolds_indices)]

# fly 0

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_sorted[file_ids.index(0)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_manifolds_sorted[file_ids.index(0)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

# fly 2

In [ ]:
# plot
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_sorted[file_ids.index(2)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_manifolds_sorted[file_ids.index(2)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

# fly 3

In [ ]:
# plot
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_sorted[file_ids.index(3)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_manifolds_sorted[file_ids.index(3)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

# fly 8


In [ ]:
# plot
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_sorted[file_ids.index(8)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_manifolds_sorted[file_ids.index(8)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

# fly 9

In [ ]:
# plot
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_sorted[file_ids.index(9)].T, -0.015, 0.015), aspect='auto', cmap='plasma')

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(np.clip(traces_manifolds_sorted[file_ids.index(9)].T, -0.015, 0.015), aspect='auto', cmap='plasma')